# CDMA Lab

This notebook has four sections:

1. **CDMA Overview** (Markdown + figures)
2. **CDMA Playground** (interactive widgets + plots)
3. **Programming Assignment** (run Python and C++ snippets)
4. **Survey**

> If widgets don’t show up:
> - `pip install ipywidgets`
> - Classic Notebook: `jupyter nbextension enable --py widgetsnbextension`

## 1. Overview

**CDMA (Code Division Multiple Access)** lets multiple users share the same time and frequency by giving each user a unique **spreading code** (chip sequence).

- **Spreading:** user bit \(b_i \in \{-1,+1\}\) times code chips \(c_i[k]\in\{-1,+1\}\) gives transmitted chips \(x_i[k]=b_i\cdot c_i[k]\).
- **Superposition:** the channel adds them: \(y[k]=\sum_i x_i[k] + n[k]\).
- **Despreading:** receiver correlates with its code: \(\hat{b}_i = \text{sign}(\sum_k y[k]\cdot c_i[k])\).
- **Key idea:** if codes have low cross-correlation, users interfere less.


### References:
[1] https://gaia.cs.umass.edu/wireless_and_mobile_networks/readings/Chapter_3_multiple_access_techniques.pdf

[2]


### Tiny visual intuition (chips vs bit)
Run the next cell to see a small animated “chips vs bit” sketch.

## 2) CDMA Playground (interactive)

Adjust the controls to see:
- received chip stream (sum + noise)
- correlation values and recovered bits

In [2]:
import numpy as np
import matplotlib.pyplot as plt

import ipywidgets as widgets
from IPython.display import display, clear_output


def rand_pm1(n):
    return np.where(np.random.rand(n) < 0.5, -1, 1)

def cdma_once(U=3, L=16, noise=0.1, bits=None, codes=None):
    if codes is None:
        codes = np.stack([rand_pm1(L) for _ in range(U)], axis=0)  # (U,L)
    if bits is None:
        bits = rand_pm1(U)  # (U,)

    tx_users = codes * bits[:, None]     # (U,L)
    sum_sig = tx_users.sum(axis=0)       # (L,)
    rx = sum_sig + noise * np.random.randn(L)

    corrs = codes @ rx                   # (U,)
    rec_bits = np.where(corrs >= 0, 1, -1)
    return codes, bits, rx, corrs, rec_bits


num_users = widgets.Dropdown(options=[2, 3, 4], value=3, description="Users:")
chip_len = widgets.Dropdown(options=[8, 16, 32], value=16, description="Chip L:")
noise = widgets.FloatSlider(value=0.10, min=0, max=1.0, step=0.01, description="Noise:")
new_codes_btn = widgets.Button(description="New codes")
random_bits_btn = widgets.Button(description="Random bits")

bits_box = widgets.HBox([])
codes_out = widgets.Textarea(value="", description="Codes:", layout=widgets.Layout(width="100%", height="120px"))
recv_out = widgets.Textarea(value="", description="Recovered:", layout=widgets.Layout(width="100%", height="120px"))
plot_out = widgets.Output(layout=widgets.Layout(width="100%"))

state = {"codes": None, "bits": None}

def rebuild_bits_ui(U):
    ds = []
    for i in range(U):
        dd = widgets.Dropdown(options=[-1, 1], value=1, description=f"U{i+1}")
        dd.observe(lambda *_: refresh(), "value")
        ds.append(dd)
    bits_box.children = ds

def get_bits_from_ui():
    return np.array([c.value for c in bits_box.children], dtype=int)

def set_bits_to_ui(bits):
    for i, b in enumerate(bits):
        bits_box.children[i].value = int(b)

def refresh(*_):
    U = int(num_users.value)
    L = int(chip_len.value)
    sig = float(noise.value)

    if state["codes"] is None or state["codes"].shape != (U, L):
        state["codes"] = np.stack([rand_pm1(L) for _ in range(U)], axis=0)

    if len(bits_box.children) != U:
        rebuild_bits_ui(U)

    if state["bits"] is None or state["bits"].shape != (U,):
        state["bits"] = rand_pm1(U)
        set_bits_to_ui(state["bits"])

    bits = get_bits_from_ui()
    state["bits"] = bits

    codes, bits, rx, corrs, rec_bits = cdma_once(U=U, L=L, noise=sig, bits=bits, codes=state["codes"])

    codes_out.value = "\n".join([f"U{i+1}: [{ ' '.join(map(str, codes[i].tolist())) }]" for i in range(U)])
    recv_out.value = "\n".join([f"U{i+1}: corr={corrs[i]:.2f}  -> bit={rec_bits[i]}" for i in range(U)])

    with plot_out:
        clear_output(wait=True)

        fig, ax = plt.subplots(figsize=(10, 2.5))
        ax.set_title("Received chips (sum + noise)")
        ax.axhline(0, lw=1)
        ax.plot(rx, marker="o")
        ax.set_xlabel("chip index")
        ax.set_ylabel("value")
        plt.show()

        fig, ax = plt.subplots(figsize=(10, 2.5))
        ax.set_title("Correlation values (per user)")
        ax.axhline(0, lw=1)
        ax.bar([f"U{i+1}" for i in range(U)], corrs)
        ax.set_ylabel("corr")
        plt.show()

def new_codes(_):
    U = int(num_users.value)
    L = int(chip_len.value)
    state["codes"] = np.stack([rand_pm1(L) for _ in range(U)], axis=0)
    refresh()

def random_bits(_):
    U = int(num_users.value)
    state["bits"] = rand_pm1(U)
    set_bits_to_ui(state["bits"])
    refresh()

num_users.observe(refresh, "value")
chip_len.observe(refresh, "value")
noise.observe(refresh, "value")
new_codes_btn.on_click(new_codes)
random_bits_btn.on_click(random_bits)

ui = widgets.VBox([
    widgets.HBox([num_users, chip_len, noise, new_codes_btn, random_bits_btn]),
    widgets.HTML("<b>User bits</b> (set each to -1 or +1):"),
    bits_box,
    widgets.HBox([codes_out, recv_out]),
    plot_out
])

display(ui)
refresh()

## 3) Programming Assignment (Python + C++)

- **Python** runs directly in this kernel.
- **C++** writes to a temp file, compiles with `g++`, then runs it.

> For C++ you need `g++` installed.

In [ ]:
import os, tempfile, subprocess
import ipywidgets as widgets
from IPython.display import display, clear_output

py_template = (
    'msg = "Hello, world!"\n'
    'print(msg)\n'
)

cpp_template = (
    '#include <iostream>\n'
    '#include <string>\n'
    'int main() {\n'
    '  std::string msg = "Hello, world!";\n'
    '  std::cout << msg << std::endl;\n'
    '  return 0;\n'
    '}\n'
)

py_editor = widgets.Textarea(value=py_template, layout=widgets.Layout(width="100%", height="260px"))
cpp_editor = widgets.Textarea(value=cpp_template, layout=widgets.Layout(width="100%", height="260px"))

out = widgets.Output(layout=widgets.Layout(width="100%"))
run_py_btn = widgets.Button(description="Run Python ▶")
run_cpp_btn = widgets.Button(description="Run C++ ▶")
reset_py_btn = widgets.Button(description="Reset")
reset_cpp_btn = widgets.Button(description="Reset")

def run_python(_):
    with out:
        clear_output()
        try:
            exec(py_editor.value, {})
        except Exception as e:
            print("Error:", e)

def run_cpp(_):
    with out:
        clear_output()
        with tempfile.TemporaryDirectory() as d:
            src = os.path.join(d, "main.cpp")
            exe = os.path.join(d, "a.out")
            with open(src, "w", encoding="utf-8") as f:
                f.write(cpp_editor.value)

            try:
                comp = subprocess.run(["g++", "-O2", src, "-o", exe], capture_output=True, text=True)
            except FileNotFoundError:
                print("g++ not found. Install a C++ compiler (e.g., g++), then try again.")
                return

            if comp.stdout.strip():
                print("--- compile stdout ---")
                print(comp.stdout)
            if comp.stderr.strip():
                print("--- compile stderr ---")
                print(comp.stderr)

            if comp.returncode != 0:
                print("\nCompilation failed.")
                return

            run = subprocess.run([exe], capture_output=True, text=True)
            if run.stdout:
                print(run.stdout, end="" if run.stdout.endswith("\n") else "\n")
            if run.stderr:
                print("\n--- stderr ---")
                print(run.stderr)

def reset_py(_): py_editor.value = py_template
def reset_cpp(_): cpp_editor.value = cpp_template

run_py_btn.on_click(run_python)
run_cpp_btn.on_click(run_cpp)
reset_py_btn.on_click(reset_py)
reset_cpp_btn.on_click(reset_cpp)

tabs = widgets.Tab(children=[
    widgets.VBox([widgets.HBox([reset_cpp_btn, run_cpp_btn]), cpp_editor]),
    widgets.VBox([widgets.HBox([reset_py_btn, run_py_btn]), py_editor]),
])
tabs.set_title(0, "C++")
tabs.set_title(1, "Python")

display(tabs, out)

Output(layout=Layout(width='100%'))

## 4) Survey
